# Linear draft win prediction

Train a linear one-hot draft model to predict the winner from heroes only. The target is `winner_side`: `0` means Radiant won, `1` means Dire won, so `sigmoid(logit)` is the model probability of a Dire win.

This is a linear model over draft features. For a binary win/loss target we train it with `BCEWithLogitsLoss`, which makes it a logistic-regression style classifier while keeping the model itself as one linear layer.

In [1]:
import importlib
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "scripts").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

import scripts.ml.dataset as dataset_module
import scripts.ml.models.linear_one_hot as linear_one_hot_module

importlib.reload(dataset_module)
importlib.reload(linear_one_hot_module)

DatasetMaker = dataset_module.DatasetMaker
LinearOneHotDraftModel = linear_one_hot_module.LinearOneHotDraftModel


In [ ]:
DOTABUFF_LIMIT = None       # None = all complete Dotabuff matches, 0 = skip
OPENDOTA_LIMIT = None          # Increase if opendota_matches is populated and should be mixed in
SYNTHETIC_LIMIT = 0         # Increase to train with synthetic_outdraft_matches

VAL_FRACTION = 0.2
BATCH_SIZE = 64
SEED = 42

LR = 3e-3
WEIGHT_DECAY = 1e-3
EPOCHS = 35
THRESHOLD = 0.5

torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


## Load drafts

The notebook uses `DatasetMaker`, so it reads database settings from `.env` in the repository root. A row is usable when both teams have exactly five heroes and `winner_side` is known.

In [ ]:
with DatasetMaker(env_file=str(PROJECT_ROOT / ".env")) as dataset:
    drafts = dataset.fetch_normalized_training_match_drafts(
        dotabuff_limit=DOTABUFF_LIMIT,
        opendota_limit=OPENDOTA_LIMIT,
        synthetic_limit=SYNTHETIC_LIMIT,
    )

complete_drafts = [
    draft
    for draft in drafts
    if draft.winner_side is not None
    and len(draft.radiant_hero_ids) == 5
    and len(draft.dire_hero_ids) == 5
]

if not complete_drafts:
    raise ValueError("No complete labeled drafts found. Check .env database settings and imported match data.")

summary = pd.DataFrame(
    {
        "matches": [len(complete_drafts)],
        "radiant_wins": [sum(d.winner_side == 0 for d in complete_drafts)],
        "dire_wins": [sum(d.winner_side == 1 for d in complete_drafts)],
        "professional_known": [sum(d.is_professional_match is not None for d in complete_drafts)],
    }
)
summary


In [ ]:
match_ids = torch.tensor([draft.match_id for draft in complete_drafts], dtype=torch.long)
radiant_ids = torch.tensor([draft.radiant_hero_ids for draft in complete_drafts], dtype=torch.long)
dire_ids = torch.tensor([draft.dire_hero_ids for draft in complete_drafts], dtype=torch.long)
labels = torch.tensor([draft.winner_side for draft in complete_drafts], dtype=torch.float32)

indices = np.arange(len(complete_drafts))
rng = np.random.default_rng(SEED)
rng.shuffle(indices)

val_size = max(1, int(len(indices) * VAL_FRACTION))
val_idx = torch.tensor(indices[:val_size], dtype=torch.long)
train_idx = torch.tensor(indices[val_size:], dtype=torch.long)

train_ds = TensorDataset(match_ids[train_idx], radiant_ids[train_idx], dire_ids[train_idx], labels[train_idx])
val_ds = TensorDataset(match_ids[val_idx], radiant_ids[val_idx], dire_ids[val_idx], labels[val_idx])

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

{
    "train_size": len(train_ds),
    "val_size": len(val_ds),
    "train_dire_win_rate": labels[train_idx].mean().item(),
    "val_dire_win_rate": labels[val_idx].mean().item(),
}


## Train the model

In [ ]:
heroes_path = PROJECT_ROOT / "dotaconstants" / "build" / "heroes.json"
heroes = json.loads(heroes_path.read_text(encoding="utf-8"))
num_heroes = max(int(hero["id"]) for hero in heroes.values()) + 1

hero_id_to_name = {
    int(hero["id"]): hero.get("localized_name") or hero["name"].removeprefix("npc_dota_hero_")
    for hero in heroes.values()
}

train_labels = labels[train_idx]
positive_count = train_labels.sum().item()
negative_count = train_labels.numel() - positive_count
pos_weight = torch.tensor([negative_count / max(positive_count, 1.0)], dtype=torch.float32, device=device)

model = LinearOneHotDraftModel(num_heroes=num_heroes).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

{
    "num_heroes": num_heroes,
    "train_radiant_wins": int(negative_count),
    "train_dire_wins": int(positive_count),
    "pos_weight": pos_weight.item(),
    "parameters": sum(p.numel() for p in model.parameters()),
}


In [ ]:
def binary_metrics_from_counts(tp, tn, fp, fn):
    total = tp + tn + fp + fn
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {
        "accuracy": (tp + tn) / total if total else 0.0,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    total_examples = 0

    for _, radiant_batch, dire_batch, label_batch in loader:
        radiant_batch = radiant_batch.to(device)
        dire_batch = dire_batch.to(device)
        label_batch = label_batch.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(radiant_batch, dire_batch)
        loss = criterion(logits, label_batch)
        loss.backward()
        optimizer.step()

        batch_size = label_batch.size(0)
        total_loss += loss.item() * batch_size
        total_examples += batch_size

    return total_loss / total_examples


@torch.no_grad()
def evaluate(model, loader, criterion, device, *, threshold=0.5):
    model.eval()
    total_loss = 0.0
    total_examples = 0
    tp = tn = fp = fn = 0

    for _, radiant_batch, dire_batch, label_batch in loader:
        radiant_batch = radiant_batch.to(device)
        dire_batch = dire_batch.to(device)
        label_batch = label_batch.to(device)

        logits = model(radiant_batch, dire_batch)
        loss = criterion(logits, label_batch)
        probabilities = torch.sigmoid(logits)
        preds = (probabilities >= threshold).long()
        labels_long = label_batch.long()

        batch_size = label_batch.size(0)
        total_loss += loss.item() * batch_size
        total_examples += batch_size

        tp += ((preds == 1) & (labels_long == 1)).sum().item()
        tn += ((preds == 0) & (labels_long == 0)).sum().item()
        fp += ((preds == 1) & (labels_long == 0)).sum().item()
        fn += ((preds == 0) & (labels_long == 1)).sum().item()

    metrics = binary_metrics_from_counts(tp, tn, fp, fn)
    metrics.update({"loss": total_loss / total_examples, "tp": tp, "tn": tn, "fp": fp, "fn": fn})
    return metrics


In [ ]:
history = []

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_metrics = evaluate(model, val_loader, criterion, device, threshold=THRESHOLD)
    row = {"epoch": epoch, "train_loss": train_loss, **{f"val_{key}": value for key, value in val_metrics.items()}}
    history.append(row)
    print(
        f"epoch={epoch:02d} "
        f"train_loss={train_loss:.4f} "
        f"val_loss={val_metrics['loss']:.4f} "
        f"val_accuracy={val_metrics['accuracy']:.3f} "
        f"val_precision={val_metrics['precision']:.3f} "
        f"val_recall={val_metrics['recall']:.3f} "
        f"val_f1={val_metrics['f1']:.3f}"
    )

history_df = pd.DataFrame(history)
history_df.tail()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

history_df.plot(x="epoch", y=["train_loss", "val_loss"], ax=axes[0])
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")

history_df.plot(x="epoch", y=["val_accuracy", "val_f1"], ax=axes[1])
axes[1].set_title("Validation metrics")
axes[1].set_xlabel("Epoch")
axes[1].set_ylim(0, 1)

plt.tight_layout()


## Compare with simple baselines

In [ ]:
val_labels = labels[val_idx]
final_metrics = evaluate(model, val_loader, criterion, device, threshold=THRESHOLD)

baseline = pd.DataFrame(
    [
        {"model": "always_radiant", "accuracy": (val_labels == 0).float().mean().item()},
        {"model": "always_dire", "accuracy": (val_labels == 1).float().mean().item()},
        {"model": "linear_draft", "accuracy": final_metrics["accuracy"], "f1": final_metrics["f1"]},
    ]
)
baseline


## Inspect hero coefficients

`draft_value = dire_weight - radiant_weight` estimates how much the logit changes when a hero is on Dire instead of Radiant. Bigger values favor the side that picks the hero; smaller values suggest the opposite in this dataset.

In [ ]:
with torch.no_grad():
    weights = model.regressor.weight.detach().cpu().squeeze(0).numpy()
    bias = model.regressor.bias.detach().cpu().item()

radiant_weights = weights[:num_heroes]
dire_weights = weights[num_heroes:]
rows = []
for hero_id, hero_name in hero_id_to_name.items():
    rows.append(
        {
            "hero_id": hero_id,
            "hero": hero_name,
            "radiant_weight": radiant_weights[hero_id],
            "dire_weight": dire_weights[hero_id],
            "draft_value": dire_weights[hero_id] - radiant_weights[hero_id],
        }
    )

coef_df = pd.DataFrame(rows).sort_values("draft_value", ascending=False)
print(f"bias={bias:.4f}")
coef_df.head(15)


In [ ]:
coef_df.tail(15).sort_values("draft_value")


## Predict a custom draft

In [ ]:
def predict_draft(radiant_hero_names, dire_hero_names):
    if len(radiant_hero_names) != 5 or len(dire_hero_names) != 5:
        raise ValueError("Both teams must contain exactly five heroes.")

    with DatasetMaker(env_file=str(PROJECT_ROOT / ".env")) as dataset:
        radiant = torch.tensor([[dataset.hero_name_to_id(hero) for hero in radiant_hero_names]], dtype=torch.long)
        dire = torch.tensor([[dataset.hero_name_to_id(hero) for hero in dire_hero_names]], dtype=torch.long)

    model.eval()
    with torch.no_grad():
        logit = model(radiant.to(device), dire.to(device)).cpu().item()
        dire_probability = 1 / (1 + np.exp(-logit))

    return {
        "radiant_win_probability": 1 - dire_probability,
        "dire_win_probability": dire_probability,
        "predicted_winner": "dire" if dire_probability >= THRESHOLD else "radiant",
    }

# Example. Replace these with any normalized or localized hero names known to dotaconstants.
predict_draft(
    radiant_hero_names=["anti mage", "crystal maiden", "axe", "puck", "lich"],
    dire_hero_names=["juggernaut", "shadow fiend", "lion", "centaur warrunner", "witch doctor"],
)


## Save model weights

In [ ]:
OUTPUT_DIR = PROJECT_ROOT / "artifacts"
OUTPUT_DIR.mkdir(exist_ok=True)

checkpoint_path = OUTPUT_DIR / "linear_draft_win_prediction.pt"
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "num_heroes": num_heroes,
        "threshold": THRESHOLD,
        "config": {
            "dotabuff_limit": DOTABUFF_LIMIT,
            "opendota_limit": OPENDOTA_LIMIT,
            "synthetic_limit": SYNTHETIC_LIMIT,
            "val_fraction": VAL_FRACTION,
            "seed": SEED,
            "lr": LR,
            "weight_decay": WEIGHT_DECAY,
            "epochs": EPOCHS,
        },
        "final_validation_metrics": final_metrics,
    },
    checkpoint_path,
)
checkpoint_path
